# Train YOLOv8n-P2 Drone V5 (Dataset Cận Cảnh ⮕ Inference Drone 3-4m)

**So với V4 (`yolov8n` cổ điển):** thêm **P2 head** — tầng feature map co ít nhất (×4, 160×160) → nhìn được **lá bệnh nhỏ vài pixel** mà P3 (80×80) bỏ sót. Nhắm đúng bài toán drone bay cao.

**P2 là gì:** backbone YOLOv8 xuất 3 feature map P3/8, P4/16, P5/32. Model này thêm nhánh **P2/4** ở tầng chi tiết nhất → cùng một lá 8px, ở P3 nằm trong 1 cell còn ở P2 nằm trong 4 cell → dễ phát hiện hơn rõ.

**Cái giá phải trả (đo thật trên ultralytics 8.4.118, `nc=39`):**

| | yolov8n (v4) | yolov8n-p2 (v5) |
|---|---|---|
| params | 3.01 M | 2.98 M |
| GFLOPs @640 | 8.2 | **13.0** |
| Detect strides | P3/8, P4/16, P5/32 | **P2/4**, P3/8, P4/16, P5/32 |

Params gần bằng nhau (P2 thêm nhánh x2 ở 32 ch), nhưng **FLOPs cao hơn ~60%** vì feature map 160×160 ở cuối backbone. Trên K230 & điện thoại chậm hơn đáng kể. Nếu FPS là ưu tiên → quay lại v4 + SAHI 320.

**⚠ Quan trọng — Vì sao không dùng pretrained `yolov8n.pt`:**
- `YOLO("yolov8n.yaml").load("yolov8n-p2.pt")` — không khả dụng: **P2 chưa có pretrained weights** (`yolov8n-p2.pt` không có trong registry ultralytics 8.4.118). Kiến trúc 4-head cũng khác nên không load được `yolov8n.pt` 3-head.
- Vì vậy notebook này train **từ đầu** từ `yolov8n-p2.yaml` (bắt buộc, không có lựa chọn khác). Lần train đầu sẽ hội tụ chậm hơn v4 (vốn khởi tạo từ COCO weights) — đây là tradeoff cố hữu của P2 trên project này.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics roboflow sahi opencv-python

from ultralytics import YOLO
import ultralytics, torch
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

# Mặc định train từ đầu bằng yolov8n-p2.yaml (vì P2 không có pretrained weights — xem cell markdown đầu).

## 1. Tải Dataset Cận Cảnh từ Roboflow

In [ ]:
def _get_roboflow_key():
    import os
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        if key: return key
    except Exception: pass
    if os.environ.get("ROBOFLOW_API_KEY"): return os.environ["ROBOFLOW_API_KEY"]
    try:
        for line in open(".env"):
            if line.startswith("ROBOFLOW_API_KEY") and "=" in line:
                return line.split("=", 1)[1].strip()
    except FileNotFoundError: pass
    raise ValueError("Thiếu ROBOFLOW_API_KEY. Thêm secret tên ROBOFLOW_API_KEY trên Kaggle!")

ROBOFLOW_API_KEY = _get_roboflow_key()
print("Đã lấy ROBOFLOW_API_KEY")

from roboflow import Roboflow
WORKSPACE       = "trantungbach26-gmail-com"
PROJECT_NAME    = "citrus-disease-detection-yoydc-ahtka"
PROJECT_VERSION = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
dataset_info = project.version(PROJECT_VERSION).download("yolov8")
print("Dataset cận cảnh đã tải về /kaggle/working/")

In [ ]:
import os, glob, yaml

candidates = glob.glob("/kaggle/working/*/data.yaml")
DATASET_PATH = os.path.dirname(candidates[0]) if candidates else "/kaggle/working/citrus-disease-detection-1"
TRAIN_DATA_YAML = os.path.join(DATASET_PATH, "data.yaml")
print("DATASET_PATH =", DATASET_PATH)

with open(TRAIN_DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print("Số class:", cfg["nc"])
print("Tên class:", cfg["names"])

## 2. Load Model N-P2

Dùng `yolov8n-p2.yaml` — 4 đầu ra P2/4, P3/8, P4/16, P5/32. **Chưa có pretrained weights** nên train từ đầu (bắt buộc).

Không cần `YOLO("yolov8n-p2.pt")` — không tồn tại trong registry; cũng không thể `load()` `yolov8n.pt` (3-head).

In [ ]:
MODEL_YAML = "yolov8n-p2.yaml"
MODEL_NAME = "yolov8n-p2"

# Tạo model từ yaml — nc được đẩy từ data yaml lúc train
model = YOLO(MODEL_YAML)
print(f"Đã load kiến trúc {MODEL_YAML} (P2/4 + P3/8 + P4/16 + P5/32) — chưa load weights, train từ đầu.")

In [ ]:
# ===== CẤU HÌNH TRAIN P2 CHO DATASET CẬN CẢNH ROBOFLOW =====
EPOCHS   = 150
IMGSZ    = 640        # Chuẩn resolution cho K230 ONNX export
BATCH    = 16         # Kích thước thực tế: params ~2.98M (nhỏ hơn v4); FLOPs 13G vs 8.2G
PATIENCE = 20

OUT_DIR = "/kaggle/working/drone_yolo_v5_p2_close_up"
os.makedirs(OUT_DIR, exist_ok=True)

# Auto-backup best.pt sau mỗi epoch
import shutil
from ultralytics.utils import callbacks

def _backup(trainer):
    try:
        src = os.path.join(trainer.save_dir, "weights", "best.pt")
        shutil.copy(src, os.path.join(OUT_DIR, "best_checkpoint.pt"))
        print(f"  [backup epoch {trainer.epoch}] -> {OUT_DIR}/best_checkpoint.pt", flush=True)
    except Exception as e:
        pass

callbacks.default_callbacks["on_fit_epoch_end"].append(_backup)

print(f"Train {MODEL_NAME}: epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, patience={PATIENCE}")

# Augmentations giúp model cận cảnh thích nghi với ảnh lá nhỏ từ xa (giống v4)
# Chú ý: P2 head vốn tăng recall vật nhỏ nhưng lúc train từ đầu cần đủ epochs để P2 ổn định.
train_args = dict(
    data=TRAIN_DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    seed=42,
    time=8,              # Tối đa 8 giờ session Kaggle
    cache=True,
    workers=2,
    # Augmentations biến đổi scale & góc chụp
    scale=0.8,           # Thu nhỏ/phóng to ngẫu nhiên từ 20% đến 180% kích thước lá
    fliplr=0.5,          # Lật ngang
    mosaic=1.0,          # Mosaic 4 ảnh cận cảnh ghép lại (tạo góc nhìn nhiều lá)
    mixup=0.15,          # Trộn ảnh tạo nhiễu ánh sáng
    copy_paste=0.2,      # Trộn vết bệnh
    cos_lr=True,         # Cosine LR decay
    project="/kaggle/working/runs",
    name="drone_yolov8n_p2_closeup",
)

results = model.train(**train_args)

## 3. Đánh giá & Export Model (pt + onnx)

File **`best.onnx`** sẽ được export để phục vụ chuyển đổi sang **`best.kmodel`** chạy trên chip Kendryte K230 của Drone.

**⚠ Lưu ý export — format khác v4 (quan trọng để không crash):**
- ONNX sau train có **4 output grid**: strides [4, 8, 16, 32], mỗi tensor kích thước khác nhau.
- Anchor tổng: (160×160 + 80×80 + 40×40 + 20×20) = 25600 + 6400 + 1600 + 400 = **34000 anchors**, không phải 8400 như v4.
- Attribute-major layout: P2 có 43 dòng (4 box + 39 class) như mọi head, nhưng tổng cột = 34000.
- Khi convert sang NCNN và viết code detect: **bắt buộc đọc cả 4 output**, hardcode stride [4,8,16,32] và total=34000. Code hiện tại của app Android + `model_ncnn.py` chặt cứng 8400 → sẽ sai/out-of-bounds nếu nạp model P2.
- Nếu export với `nms=True`: nhập lại logic khớp stride tương tự.

→ **Trước khi deploy P2 lên app/K230 cần cập nhật code detect**; hoặc export thêm bản strip-P2 (bỏ nhánh P2) nếu muốn giữ nguyên code cũ nhưng khi đó mất đúng ý nghĩa của P2.

In [ ]:
metrics = model.val()
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

In [ ]:
# Export ONNX dành cho chip K230 Drone
RESULTS_DIR = "/kaggle/working/runs/drone_yolov8n_p2_closeup/weights"
best_path   = os.path.join(RESULTS_DIR, "best.pt")

if os.path.exists(best_path):
    best_model = YOLO(best_path)
    best_model.export(format="onnx", imgsz=IMGSZ, opset=11, simplify=True)
    shutil.copy(best_path, os.path.join(OUT_DIR, "best.pt"))
    onnx_src = os.path.join(RESULTS_DIR, "best.onnx")
    if os.path.exists(onnx_src):
        shutil.copy(onnx_src, os.path.join(OUT_DIR, "best.onnx"))
    print(f"Đã copy best.pt và best.onnx vào {OUT_DIR}")
else:
    print(f"Không tìm thấy {best_path}")

print("\n>>> TẢI KẾT QUẢ: Panel bên phải tab 'Output' -> biểu tượng Download all.")

## Ghi chú sau khi train xong

- **So sánh với v4:** P2 train từ đầu (không COCO-pretrained) nên mAP50 có thể THẤP hơn v4 trong cùng 8h cap — đừng vội kết luận P2 dở. Nhìn kỹ `metrics.box.mr` (recall, nhất là trên vết bệnh nhỏ) và chạy `tile_and_detect.py` / SAHI 320 trên ảnh drone thật. Nếu recall vật nhỏ tăng → P2 thành công dù mAP50 cao toàn cục chưa bằng v4.
- **Precision:** augment mạnh (fliplr/mixup/copy_paste) có thể làm precision tụt nhẹ so với v4. Kiểm tra bằng `model.val()` + xem false positive trên ảnh drone. Nếu cần tăng precision: hạ `conf` lúc chạy hoặc giảm `scale` về 0.6.
- **Nếu muốn dùng trên Android app hiện tại:** bắt buộc cập nhật `yolov8_det.cpp` để đọc 4 output + stride [4,8,16,32] + tổng 34000 anchors — hoặc export thêm bản strip-P2 để giữ nguyên code cũ (nhưng mất ý nghĩa P2).
- **K230:** convert kmodel từ ONNX này tương tự v1/v2. Với v5 (P2) thì FLOPs 13G cao hơn — kiểm tra RAM NPU / độ trễ trước khi deploy.
- **Nếu muốn P2 nhưng FPS không chịu nổi:** chạy model P2 với imgsz 448 (thay 640) khi export/inference — nhánh P2 nhẹ hơn nhiều, hao chút recall vật nhỏ.